In [3]:
uv pip install pydantic

Note: you may need to restart the kernel to use updated packages.


C:\Users\sharm\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe: No module named uv


In [6]:
import sys
print(sys.executable)

C:\Users\sharm\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe


In [5]:
from typing import Literal
#from pydantic import BaseModel, Field 
#from langchain_openai import ChatOpenAI

print("All imports working!")

All imports working!


In [ ]:
uv pip install py~dantic langchain langchain-openai

Note: you may need to restart the kernel to use updated packages.


C:\Users\sharm\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe: No module named uv


In [12]:
from typing import Literal
# from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

class ProposedAction(BaseModel):
    action: Literal["search", "answer", "clarify"] = Field(
        description="The action the model should take"
    )
    reason: str = Field(
        description="Why this action was selected"
    )

model = ChatOpenAI(model="gpt-4o-mini")

structured_model = model.with_structured_output(ProposedAction)

response = structured_model.invoke(
    "I don't understand how LangGraph works."
)

print(response)

ModuleNotFoundError: No module named 'langchain_openai'

In [18]:
structured_model = model.with_structured_output(ProposedAction)

NameError: name 'model' is not defined

In [ ]:
structured_model.invoke("""Customer CUST-202 is asking for a ₹5,000 refund,
but the purchase was made 8 months ago and is outside
the refund policy.""")

In [ ]:
structured_model.invoke("""Customer CUST-303 is requesting a ₹75,000 refund.
The customer claims the product was defective,
but the available information is insufficient to verify the claim.""")

In [ ]:
structured_model.invoke("""Customer CUST-501 ordered a laptop for ₹65,000.
The order was cancelled before shipment,
but the payment has already been deducted.
Please process the refund.""")

In [ ]:
def ai_assistant(user_request):

    prompt = f"""
You are a customer support AI agent.

Analyze the following customer request:

{user_request}

Decide the appropriate action.

Possible actions:
- refund
- reject
- manual_review

Return:
- action
- customer ID
- refund amount
- reason

Do not execute the action.
Only propose the action for human approval.
"""

    proposed_action = structured_model.invoke(
        prompt
    )

    return proposed_action

In [ ]:
def human_review(action):

    print("\n========== AI Proposed Action ==========")

    print("Action   :", action.action)
    print("Customer :", action.customer)
    print("Amount   :", action.amount)
    print("Reason   :", action.reason)

    print("========================================")


    decision = input(
        "\nApprove this action? (yes/no): "
    ).strip().lower()


    if decision == "yes":
        return True

    return False


In [ ]:
def execute_action(action):

    if action.action == "refund":

        print(
            f"\nRefund of ₹{action.amount} "
            f"processed for {action.customer}."
        )


    elif action.action == "reject":

        print(
            f"\nRequest rejected for "
            f"{action.customer}."
        )


    elif action.action == "manual_review":

        print(
            f"\nRequest for {action.customer} "
            f"sent for manual review."
        )

In [ ]:
def run():

    user_request = input(
        "User Request: "
    )


    # Step 1
    # LLM analyzes request and proposes action

    proposed_action = ai_assistant(
        user_request
    )


    # Step 2
    # Human reviews the AI decision

    approved = human_review(
        proposed_action
    )


    # Step 3
    # Execute only after human approval

    if approved:

        execute_action(
            proposed_action
        )

    else:

        print(
            "\nAction rejected by human."
        )


In [ ]:
run()

In [19]:
from typing import TypedDict, Literal

from pydantic import BaseModel

from langchain_openai import ChatOpenAI

from langgraph.graph import (
    StateGraph,
    START,
    END
)

from langgraph.checkpoint.memory import (
    InMemorySaver
)

from langgraph.types import (
    interrupt,
    Command
)

ModuleNotFoundError: No module named 'pydantic'

In [ ]:
model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

In [ ]:
class ProposedAction(BaseModel):

    action: Literal[
        "refund",
        "reject",
        "manual_review"
    ]

    customer: str

    amount: float

    reason: str

In [ ]:
structured_model = model.with_structured_output(
    ProposedAction
)

In [ ]:
class AgentState(TypedDict):
    user_request: str

    action: str

    customer: str

    amount: float

    reason: str

    approved: bool

    final_message: str

In [ ]:
def analyze_request(state: AgentState):

    prompt = f"""
You are a customer support AI agent.

Analyze the following customer request:

{state["user_request"]}

Decide the appropriate action.

Possible actions:
- refund
- reject
- manual_review

Return:
- action
- customer
- amount
- reason

Do not execute anything.
Only propose an action.
"""


    result = structured_model.invoke(
        prompt
    )


    return {
        "action": result.action,
        "customer": result.customer,
        "amount": result.amount,
        "reason": result.reason
    }

In [ ]:
def human_approval(state: AgentState):

    # Graph pauses here

    decision = interrupt(
        {
            "message": "Please review the AI proposed action.",

            "action": state["action"],

            "customer": state["customer"],

            "amount": state["amount"],

            "reason": state["reason"]
        }
    )


    # decision comes from Command(resume=...)
    return {
        "approved": decision
    }

In [ ]:
def approval_router(
    state: AgentState
):

    if state["approved"]:
        return "execute"

    return "reject"

In [ ]:
def execute_action(
    state: AgentState
):

    if state["action"] == "refund":

        message = (
            f"Refund of ₹{state['amount']} "
            f"processed for "
            f"{state['customer']}."
        )


    elif state["action"] == "reject":

        message = (
            f"Request rejected for "
            f"{state['customer']}."
        )


    else:

        message = (
            f"Request for "
            f"{state['customer']} "
            f"sent for manual review."
        )


    return {
        "final_message": message
    }

In [ ]:
def reject_action(
    state: AgentState
):

    return {
        "final_message":
        "Action rejected by human reviewer."
    }

In [ ]:
builder = StateGraph(
    AgentState
)

In [ ]:
builder.add_node(
    "analyze",
    analyze_request
)


builder.add_node(
    "human_approval",
    human_approval
)


builder.add_node(
    "execute",
    execute_action
)


builder.add_node(
    "reject",
    reject_action
)

In [ ]:
builder.add_edge(
    START,
    "analyze"
)


builder.add_edge(
    "analyze",
    "human_approval"
)


builder.add_conditional_edges(
    "human_approval",
    approval_router,
    {
        "execute": "execute",
        "reject": "reject"
    }
)


builder.add_edge(
    "execute",
    END
)


builder.add_edge(
    "reject",
    END
)


In [ ]:
checkpointer = InMemorySaver()

In [ ]:
graph = builder.compile(
    checkpointer=checkpointer
)

In [ ]:
graph

In [ ]:
config = {
    "configurable": {
        "thread_id": "refund_thread_1"
    }
}

In [ ]:
result = graph.invoke(

    {
        "user_request":
        """
Customer CUST-101 received a damaged product.
Please refund ₹5000.
""",

        "action": "",
        "customer": "",
        "amount": 0,
        "reason": "",
        "approved": False,
        "final_message": ""
    },

    config=config
)

In [ ]:
print(
    "\nGraph paused for human approval."
)


print(
    "\nInterrupt Information:"
)


print(
    result["__interrupt__"]
)

In [ ]:
human_input = input(
    "\nApprove action? (yes/no): "
).strip().lower()

In [ ]:
human_input

In [ ]:
approved = (
    human_input == "yes"
)

In [ ]:
approved

In [ ]:
final_result = graph.invoke(

    Command(
        resume=approved
    ),

    config=config
)

In [ ]:
print(
    "\nFinal Result:"
)

print(
    final_result["final_message"]
)